# LeetCode 1863 — Sum of All Subset XOR Totals bằng Python

Link đề: https://leetcode.com/problems/sum-of-all-subset-xor-totals/

Độ khó: **Easy** · Chủ đề: Array, Math, Backtracking, Bit Manipulation, Combinatorics, Enumeration

Skeleton chữ ký LeetCode:

```python
class Solution:
    def subsetXORSum(self, nums: List[int]) -> int:

```

**Mục tiêu học**

1. Hiểu phép **XOR** (`^`) trên từng bit và các tính chất: `a ^ a = 0`, `a ^ 0 = a`, giao hoán, kết hợp.
2. Biết cách **liệt kê mọi tập con** (subset) của mảng: bằng **bitmask** và bằng **đệ quy chọn / không chọn** (backtracking).
3. Học thêm cách xây dần danh sách XOR của các tập con (lặp, không đệ quy).
4. Khám phá mẹo toán học: đáp án = `(OR của mọi phần tử) × 2^(n-1)` — và **chứng minh** vì sao.
5. Trace, Big-O `O(n·2^n)` → `O(2^n)` → `O(n)`, lỗi hay gặp và 6 bài tập về tập con / bit.

## 1. Bài toán yêu cầu gì?

**XOR total** của một mảng = XOR của mọi phần tử, hoặc `0` nếu mảng rỗng.

Cho `nums`, trả về **tổng XOR total của mọi tập con** của `nums`. Các tập con có phần tử giống nhau (nhưng lấy từ vị trí khác nhau) vẫn được tính **riêng**.

Một tập con ở đây = xóa đi một số (có thể 0) phần tử — thứ tự không quan trọng với XOR.

| nums | Kết quả |
|---|---:|
| `[1,3]` | `0 + 1 + 3 + (1^3=2) = 6` |
| `[5,1,6]` | `0+5+1+6+4+3+7+2 = 28` |
| `[3,4,5,6,7,8]` | `480` |

**Constraints:** `1 <= len(nums) <= 12`, `1 <= nums[i] <= 20`.

`n <= 12` ⇒ có tối đa `2^12 = 4096` tập con — đủ nhỏ để **liệt kê hết**.

In [1]:
from typing import List

examples = [([1, 3], 6), ([5, 1, 6], 28), ([3, 4, 5, 6, 7, 8], 480)]
for nums, expected in examples:
    print(nums, "->", expected, "| số tập con =", 2 ** len(nums))

[1, 3] -> 6 | số tập con = 4
[5, 1, 6] -> 28 | số tập con = 8
[3, 4, 5, 6, 7, 8] -> 480 | số tập con = 64


## 2. Nền tảng: bit và XOR

### 2.1. Số nhị phân và `bin()`

Mỗi số nguyên không âm viết được ở cơ số 2. `bin(5) = '0b101'` nghĩa là `5 = 4 + 1`. `format(x, '04b')` in đủ 4 bit.

In [2]:
for x in [1, 3, 5, 6]:
    print(x, "=", bin(x), "=", format(x, "04b"))

1 = 0b1 = 0001
3 = 0b11 = 0011
5 = 0b101 = 0101
6 = 0b110 = 0110


### 2.2. XOR: "khác nhau thì 1, giống nhau thì 0"

`a ^ b` so từng cặp bit: bit kết quả là 1 nếu **đúng một** trong hai bit là 1.

```
  5 = 101
^ 6 = 110
------------
  3 = 011
```

In [3]:
a, b = 5, 6
print(f"{a:03b} ^ {b:03b} = {a ^ b:03b} = {a ^ b}")
print("bảng chân trị:")
for x in (0, 1):
    for y in (0, 1):
        print(f"  {x} ^ {y} = {x ^ y}")

101 ^ 110 = 011 = 3
bảng chân trị:
  0 ^ 0 = 0
  0 ^ 1 = 1
  1 ^ 0 = 1
  1 ^ 1 = 0


### 2.3. Các tính chất XOR cần nhớ

- `a ^ 0 = a` (0 là phần tử trung hòa — vì vậy tập rỗng có XOR = 0).
- `a ^ a = 0` (tự triệt tiêu).
- Giao hoán `a ^ b = b ^ a`, kết hợp `(a ^ b) ^ c = a ^ (b ^ c)` ⇒ thứ tự XOR không quan trọng.

In [4]:
a, b, c = 5, 1, 6
print("a ^ 0 =", a ^ 0, "| a ^ a =", a ^ a)
print("(a^b)^c =", (a ^ b) ^ c, "| a^(b^c) =", a ^ (b ^ c), "| c^a^b =", c ^ a ^ b)

a ^ 0 = 5 | a ^ a = 0
(a^b)^c = 2 | a^(b^c) = 2 | c^a^b = 2


### 2.4. XOR của cả list bằng `functools.reduce`

`reduce(f, [x1, x2, x3], start)` tính `f(f(f(start, x1), x2), x3)`. Với `operator.xor` ta được XOR của cả list.

In [5]:
from functools import reduce
from operator import xor

print(reduce(xor, [2, 5, 6], 0))      # 2 ^ 5 ^ 6 = 1 (ví dụ trong đề)
print(reduce(xor, [], 0))             # tập rỗng -> 0

1
0


### 2.5. Bitmask: dùng một số để mô tả một tập con

Với `n` phần tử, mỗi số `mask` từ `0` tới `2^n - 1` ứng với một tập con: bit thứ `i` của `mask` bằng 1 ⇔ **chọn** `nums[i]`.

- `1 << n` = `2^n`.
- `(mask >> i) & 1` lấy bit thứ `i` của `mask` (`>>` dịch phải, `& 1` lấy bit cuối).

In [6]:
nums = [5, 1, 6]
n = len(nums)
print("1 << n =", 1 << n)
for mask in range(1 << n):
    chosen = [nums[i] for i in range(n) if (mask >> i) & 1]
    print(f"mask={mask} ({mask:03b}) -> chọn {chosen}")

1 << n = 8
mask=0 (000) -> chọn []
mask=1 (001) -> chọn [5]
mask=2 (010) -> chọn [1]
mask=3 (011) -> chọn [5, 1]
mask=4 (100) -> chọn [6]
mask=5 (101) -> chọn [5, 6]
mask=6 (110) -> chọn [1, 6]
mask=7 (111) -> chọn [5, 1, 6]


Bit **phải nhất** ứng với `nums[0]`. Ví dụ `mask = 5 = 101` chọn `nums[0]` và `nums[2]`.

## 3. Cách 1 — liệt kê bằng bitmask (baseline)

Với mỗi `mask`, tính XOR các phần tử được chọn rồi cộng vào tổng.

In [7]:
def subset_xor_sum_bitmask(nums: List[int]) -> int:
    n = len(nums)
    total = 0
    for mask in range(1 << n):
        x = 0
        for i in range(n):
            if (mask >> i) & 1:
                x ^= nums[i]
        total += x
    return total

for nums, expected in examples:
    print(nums, "->", subset_xor_sum_bitmask(nums), "| expected", expected)
    assert subset_xor_sum_bitmask(nums) == expected

[1, 3] -> 6 | expected 6
[5, 1, 6] -> 28 | expected 28
[3, 4, 5, 6, 7, 8] -> 480 | expected 480


`x ^= v` là viết tắt `x = x ^ v`.

**Chi phí:** `2^n` mask × `n` bit ⇒ `O(n·2^n)`. Với `n = 12`: ≈ 49.000 phép — rất nhanh. Nhưng ta lặp lại việc XOR từ đầu cho mỗi tập con. Có cách **tái sử dụng** kết quả không?

## 4. Cách 2 — đệ quy "chọn / không chọn" (backtracking)

Hình dung cây quyết định: ở phần tử `i`, có 2 nhánh — **không chọn** `nums[i]` (giữ `cur`) hoặc **chọn** (`cur ^ nums[i]`). Đi tới hết mảng (`i == n`) là có một tập con, trả về `cur`.

`dfs(i, cur)` = tổng XOR total của mọi tập con mà phần đầu `nums[0..i-1]` đã quyết định và cho XOR là `cur`.

In [8]:
def subset_xor_sum_dfs(nums: List[int]) -> int:
    n = len(nums)

    def dfs(i: int, cur: int) -> int:
        if i == n:                    # đã quyết định hết -> một tập con
            return cur
        skip = dfs(i + 1, cur)                # không chọn nums[i]
        take = dfs(i + 1, cur ^ nums[i])      # chọn nums[i]
        return skip + take

    return dfs(0, 0)

for nums, expected in examples:
    assert subset_xor_sum_dfs(nums) == expected
print(subset_xor_sum_dfs([5, 1, 6]))

28


Hàm `dfs` định nghĩa **bên trong** hàm khác (hàm lồng) đọc được biến `nums`, `n` của hàm ngoài — gọi là **closure**.

Mỗi tập con chỉ tốn `O(1)` thêm ở lá vì `cur` được **truyền xuống** chứ không tính lại ⇒ tổng `O(2^n)` lời gọi.

## 5. TRACE cây đệ quy

Trace `nums = [1, 3]`. Cây quyết định:

```
                dfs(0, 0)
           /                \
   bỏ 1: dfs(1, 0)      lấy 1: dfs(1, 1)
     /        \            /        \
 bỏ 3       lấy 3       bỏ 3       lấy 3
dfs(2,0)   dfs(2,3)   dfs(2,1)   dfs(2,1^3=2)
  = 0        = 3        = 1         = 2
```

Tổng các lá: `0 + 3 + 1 + 2 = 6`.

In [9]:
def trace_dfs(nums):
    n = len(nums)

    def dfs(i, cur, depth, chosen):
        pad = "    " * depth
        if i == n:
            print(f"{pad}lá: tập {chosen} -> XOR = {cur}")
            return cur
        print(f"{pad}dfs(i={i}, cur={cur}) xét nums[{i}]={nums[i]}")
        skip = dfs(i + 1, cur, depth + 1, chosen)
        take = dfs(i + 1, cur ^ nums[i], depth + 1, chosen + [nums[i]])
        print(f"{pad}=> dfs(i={i}, cur={cur}) = {skip} + {take} = {skip + take}")
        return skip + take

    return dfs(0, 0, 0, [])

assert trace_dfs([1, 3]) == 6

dfs(i=0, cur=0) xét nums[0]=1
    dfs(i=1, cur=0) xét nums[1]=3
        lá: tập [] -> XOR = 0
        lá: tập [3] -> XOR = 3
    => dfs(i=1, cur=0) = 0 + 3 = 3
    dfs(i=1, cur=1) xét nums[1]=3
        lá: tập [1] -> XOR = 1
        lá: tập [1, 3] -> XOR = 2
    => dfs(i=1, cur=1) = 1 + 2 = 3
=> dfs(i=0, cur=0) = 3 + 3 = 6


Trace dạng bảng cho bitmask `nums = [5, 1, 6]` — cột "XOR" tính dồn:

| mask | bit (n2 n1 n0) | tập con | XOR | tổng dồn |
|---:|:---:|---|---:|---:|
| 0 | 000 | {} | 0 | 0 |
| 1 | 001 | {5} | 5 | 5 |
| 2 | 010 | {1} | 1 | 6 |
| 3 | 011 | {5,1} | 4 | 10 |
| 4 | 100 | {6} | 6 | 16 |
| 5 | 101 | {5,6} | 3 | 19 |
| 6 | 110 | {1,6} | 7 | 26 |
| 7 | 111 | {5,1,6} | 2 | 28 |

In [10]:
def trace_bitmask(nums):
    n, total = len(nums), 0
    print("mask | bit | tập con    | XOR | tổng dồn")
    for mask in range(1 << n):
        chosen = [nums[i] for i in range(n) if (mask >> i) & 1]
        x = reduce(xor, chosen, 0)
        total += x
        print(f"{mask:>4} | {mask:0{n}b} | {str(chosen):<10} | {x:>3} | {total:>4}")
    return total

assert trace_bitmask([5, 1, 6]) == 28

mask | bit | tập con    | XOR | tổng dồn
   0 | 000 | []         |   0 |    0
   1 | 001 | [5]        |   5 |    5
   2 | 010 | [1]        |   1 |    6
   3 | 011 | [5, 1]     |   4 |   10
   4 | 100 | [6]        |   6 |   16
   5 | 101 | [5, 6]     |   3 |   19
   6 | 110 | [1, 6]     |   7 |   26
   7 | 111 | [5, 1, 6]  |   2 |   28


`f"{mask:0{n}b}"` — độ rộng `n` lấy từ biến, f-string cho phép lồng `{}` trong định dạng.

## 6. Cách 3 — xây dần list XOR của các tập con (lặp)

Bắt đầu `xors = [0]` (chỉ có tập rỗng). Với mỗi `v`, mỗi tập con cũ sinh thêm một tập con mới có `v`: `xors = xors + [x ^ v for x in xors]`. Sau `n` bước có đủ `2^n` giá trị.

In [11]:
def subset_xor_sum_iterative(nums: List[int]) -> int:
    xors = [0]
    for v in nums:
        xors += [x ^ v for x in xors]
    return sum(xors)

xors = [0]
for v in [5, 1, 6]:
    xors += [x ^ v for x in xors]
    print(f"sau khi thêm {v}: {xors}")
print("tổng =", sum(xors))

sau khi thêm 5: [0, 5]
sau khi thêm 1: [0, 5, 1, 4]
sau khi thêm 6: [0, 5, 1, 4, 6, 3, 7, 2]
tổng = 28


Lưu ý: `[x ^ v for x in xors]` được tính **xong** từ list cũ rồi mới nối vào, nên không bị lặp vô hạn.

## 7. Insight toán học — OR × 2^(n−1)

Quan sát kết quả: `[1,3]`: `1 | 3 = 3`, `3 × 2^1 = 6` ✓. `[5,1,6]`: `5|1|6 = 7`, `7 × 2^2 = 28` ✓.

**Vì sao?** Xét riêng **một bit** `b` (ví dụ bit giá trị 4):

- Nếu **không** phần tử nào có bit `b` ⇒ mọi XOR total đều có bit `b` = 0 ⇒ đóng góp 0.
- Nếu **có ít nhất một** phần tử `p` có bit `b`: ghép cặp mỗi tập con `S` không chứa `p` với `S ∪ {p}`. Trong mỗi cặp, đúng **một** tập có bit `b` bằng 1 (vì thêm `p` lật bit `b`). Có `2^(n-1)` cặp ⇒ đúng `2^(n-1)` tập con có bit `b` = 1 ⇒ đóng góp `2^b × 2^(n-1)`.

Cộng mọi bit: đáp án = `(tổng các bit xuất hiện) × 2^(n-1)` = `(OR mọi phần tử) × 2^(n-1)`.

In [12]:
nums = [5, 1, 6]
n = len(nums)
for bit in range(3):
    has = [v for v in nums if (v >> bit) & 1]
    cnt = 0
    for mask in range(1 << n):
        x = reduce(xor, [nums[i] for i in range(n) if (mask >> i) & 1], 0)
        cnt += (x >> bit) & 1
    print(f"bit {bit} (giá trị {1 << bit}): phần tử có bit = {has}, số tập con có bit này = {cnt} (2^(n-1) = {2 ** (n - 1)})")

bit 0 (giá trị 1): phần tử có bit = [5, 1], số tập con có bit này = 4 (2^(n-1) = 4)
bit 1 (giá trị 2): phần tử có bit = [6], số tập con có bit này = 4 (2^(n-1) = 4)
bit 2 (giá trị 4): phần tử có bit = [5, 6], số tập con có bit này = 4 (2^(n-1) = 4)


In [13]:
def subset_xor_sum_or(nums: List[int]) -> int:
    acc = 0
    for v in nums:
        acc |= v
    return acc << (len(nums) - 1)     # nhân 2^(n-1)

for nums, expected in examples:
    assert subset_xor_sum_or(nums) == expected
print(subset_xor_sum_or([3, 4, 5, 6, 7, 8]), "| 3|4|...|8 =", reduce(lambda a, b: a | b, [3, 4, 5, 6, 7, 8]))

480 | 3|4|...|8 = 15


`acc << k` = `acc × 2^k`. `|` là OR: bit kết quả là 1 nếu **ít nhất một** bit là 1.

## 8. Code nộp LeetCode

### Cách A — backtracking (thể hiện kỹ năng liệt kê tập con)

In [14]:
class Solution:
    def subsetXORSum(self, nums: List[int]) -> int:
        n = len(nums)

        def dfs(i: int, cur: int) -> int:
            if i == n:
                return cur
            return dfs(i + 1, cur) + dfs(i + 1, cur ^ nums[i])

        return dfs(0, 0)

print([Solution().subsetXORSum(nums) for nums, _ in examples])

[6, 28, 480]


### Cách B — bitmask

In [15]:
class SolutionBitmask:
    def subsetXORSum(self, nums: List[int]) -> int:
        n, total = len(nums), 0
        for mask in range(1 << n):
            x = 0
            for i in range(n):
                if mask >> i & 1:
                    x ^= nums[i]
            total += x
        return total

print([SolutionBitmask().subsetXORSum(nums) for nums, _ in examples])

[6, 28, 480]


Thứ tự ưu tiên: `>>` cao hơn `&`, nên `mask >> i & 1` = `(mask >> i) & 1`. Viết ngoặc vẫn dễ đọc hơn.

### Cách C — công thức OR (O(n))

In [16]:
class SolutionOr:
    def subsetXORSum(self, nums: List[int]) -> int:
        acc = 0
        for v in nums:
            acc |= v
        return acc << (len(nums) - 1)

print([SolutionOr().subsetXORSum(nums) for nums, _ in examples])

[6, 28, 480]


## 9. Vì sao đúng?

**Bitmask:** ánh xạ `mask ↔ tập con` là **song ánh** giữa `{0, …, 2^n − 1}` và mọi tập con (mỗi phần tử ↔ một bit chọn/không). Nên vòng `mask` duyệt **mỗi tập con đúng một lần**.

**Backtracking:** Invariant — `dfs(i, cur)` trả về tổng XOR total của mọi cách quyết định `nums[i..n-1]`, cộng với phần đã quyết là `cur`. Base `i == n`: chỉ một cách (không còn gì), XOR = `cur`. Bước: mọi tập con hoặc **không** chứa `nums[i]`, hoặc **có** — hai nhánh rời nhau và phủ hết.

**Công thức OR:** đã chứng minh ở mục 7 bằng ghép cặp `S ↔ S ∪ {p}`.

In [17]:
# Kiểm chứng song ánh: các tập chỉ số sinh ra từ mask là khác nhau và đủ 2^n
n = 4
subsets = {tuple(i for i in range(n) if (m >> i) & 1) for m in range(1 << n)}
print("số tập con khác nhau:", len(subsets), "= 2^n =", 2 ** n)

số tập con khác nhau: 16 = 2^n = 16


## 10. Độ phức tạp

Gọi `n = len(nums)` (≤ 12).

| Cách | Time | Space |
|---|---|---|
| Bitmask | `O(n · 2^n)` | `O(1)` |
| Backtracking | `O(2^n)` lời gọi | `O(n)` (độ sâu đệ quy) |
| List XOR lặp | `O(2^n)` | `O(2^n)` (list) |
| Công thức OR | `O(n)` | `O(1)` |

Với `n = 12`: `2^12 = 4096` — mọi cách đều tức thì. Công thức OR là "mẹo đẹp", còn liệt kê tập con là **kỹ năng tổng quát** dùng cho nhiều bài khác.

In [18]:
import time
nums12 = list(range(1, 13))
for f in [subset_xor_sum_bitmask, subset_xor_sum_dfs, subset_xor_sum_iterative, subset_xor_sum_or]:
    t = time.perf_counter()
    r = f(nums12)
    print(f"{f.__name__:<26} = {r} ({(time.perf_counter() - t) * 1000:.2f} ms, tùy máy)")

subset_xor_sum_bitmask     = 30720 (2.69 ms, tùy máy)
subset_xor_sum_dfs         = 30720 (0.46 ms, tùy máy)
subset_xor_sum_iterative   = 30720 (0.15 ms, tùy máy)
subset_xor_sum_or          = 30720 (0.00 ms, tùy máy)


## 11. Bộ test

Nhóm: mẫu đề; một phần tử; phần tử trùng nhau (vẫn đếm riêng); mọi phần tử giống nhau; `n = 12` lớn nhất; ngẫu nhiên đối chiếu 7 cài đặt.

In [19]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS ({actual})")

sol = Solution()
check("mẫu 1", sol.subsetXORSum([1, 3]), 6)
check("mẫu 2", sol.subsetXORSum([5, 1, 6]), 28)
check("mẫu 3", sol.subsetXORSum([3, 4, 5, 6, 7, 8]), 480)
check("một phần tử", sol.subsetXORSum([7]), 7)
check("trùng [2,2]", sol.subsetXORSum([2, 2]), 0 + 2 + 2 + 0)
check("toàn 1 (n=3)", sol.subsetXORSum([1, 1, 1]), 4)
check("n=12 toàn 20", sol.subsetXORSum([20] * 12), 20 * 2 ** 11)

mẫu 1                  PASS (6)
mẫu 2                  PASS (28)
mẫu 3                  PASS (480)
một phần tử            PASS (7)
trùng [2,2]            PASS (4)
toàn 1 (n=3)           PASS (4)
n=12 toàn 20           PASS (40960)


In [20]:
import random
rng = random.Random(1863)
impls = [subset_xor_sum_bitmask, subset_xor_sum_dfs, subset_xor_sum_iterative, subset_xor_sum_or,
         Solution().subsetXORSum, SolutionBitmask().subsetXORSum, SolutionOr().subsetXORSum]
for _ in range(400):
    nums = [rng.randint(1, 20) for _ in range(rng.randint(1, 10))]
    res = {f(nums) for f in impls}
    assert len(res) == 1, (nums, res)
print("7 cài đặt đồng ý trên 400 mảng ngẫu nhiên PASS")

7 cài đặt đồng ý trên 400 mảng ngẫu nhiên PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: dùng `^` mà tưởng là lũy thừa (hoặc ngược lại dùng `**` thay XOR)

In [21]:
print("2 ^ 3 =", 2 ^ 3, "(XOR)  | 2 ** 3 =", 2 ** 3, "(lũy thừa)")
def wrong_power_count(nums):
    return reduce(lambda a, b: a | b, nums) * 2 ^ (len(nums) - 1)    # ^ không phải mũ!
print("sai :", wrong_power_count([5, 1, 6]), "| đúng:", subset_xor_sum_or([5, 1, 6]))

2 ^ 3 = 1 (XOR)  | 2 ** 3 = 8 (lũy thừa)
sai : 12 | đúng: 28


Sửa: `2 ** (n - 1)` hoặc `1 << (n - 1)`. Thêm nữa `*` ưu tiên cao hơn `^`, nên biểu thức trên còn bị nhóm sai.

### Lỗi 2: dùng `set` để "loại trùng" tập con

In [22]:
def wrong_unique_values(nums):
    total = 0
    vals = list(set(nums))           # bỏ phần tử trùng -> mất tập con!
    for mask in range(1 << len(vals)):
        total += reduce(xor, [vals[i] for i in range(len(vals)) if mask >> i & 1], 0)
    return total

print("sai :", wrong_unique_values([2, 2]), "| đúng:", subset_xor_sum_bitmask([2, 2]))

sai : 2 | đúng: 4


Đề: tập con có phần tử giống nhau vẫn **đếm riêng**. Sửa: liệt kê theo **chỉ số**, không theo giá trị.

### Lỗi 3: dùng `|` (OR) thay `^` khi tính XOR của tập con

In [23]:
def wrong_or_inside(nums):
    total = 0
    for mask in range(1 << len(nums)):
        x = 0
        for i in range(len(nums)):
            if mask >> i & 1:
                x |= nums[i]
        total += x
    return total

print("sai :", wrong_or_inside([1, 3]), "| đúng:", subset_xor_sum_bitmask([1, 3]))

sai : 7 | đúng: 6


`1 | 3 = 3` nhưng `1 ^ 3 = 2`. OR chỉ xuất hiện trong **công thức tắt** ở cấp toàn mảng, không phải trong từng tập con.

### Lỗi 4: quên tập rỗng hoặc trả về sai ở base case đệ quy

In [24]:
def wrong_base_case(nums):
    n = len(nums)
    def dfs(i, cur):
        if i == n:
            return 0                  # quên trả cur -> mọi lá đều 0
        return dfs(i + 1, cur) + dfs(i + 1, cur ^ nums[i])
    return dfs(0, 0)

print("sai :", wrong_base_case([1, 3]), "| đúng:", subset_xor_sum_dfs([1, 3]))

sai : 0 | đúng: 6


Sửa: ở lá trả về `cur` — giá trị XOR của tập con vừa hoàn thành. (Tập rỗng có `cur = 0`, đóng góp 0, nên không cần xử lý riêng.)

## 13. Mẹo nhận dạng pattern

- `n <= 15~20` + "mọi tập con" ⇒ **liệt kê `2^n`** bằng bitmask hoặc backtracking.
- Mỗi phần tử có **2 lựa chọn** (lấy/bỏ, +/−) ⇒ cây đệ quy nhị phân `dfs(i, state)`.
- Câu hỏi về **tổng qua mọi tập con** của một phép bit ⇒ nghĩ theo **từng bit độc lập** và đếm số tập con làm bit đó bằng 1.
- XOR: nhớ `a ^ a = 0`, `a ^ 0 = a`, thứ tự không quan trọng.
- Thấy `^` trong code Python ⇒ đó là XOR, **không phải** lũy thừa.

## 14. Biến thể

1. **LeetCode 78 — Subsets**: trả về danh sách mọi tập con.
2. **Tổng của tổng mọi tập con** (thay XOR bằng `+`): mỗi phần tử xuất hiện ở `2^(n-1)` tập ⇒ `sum(nums) × 2^(n-1)`.
3. **Tổng OR của mọi tập con**: theo từng bit, số tập con có bit = 1 là `2^n − 2^(n − c)` (c = số phần tử có bit đó).
4. **Đếm tập con có XOR = k**: liệt kê hoặc DP trên giá trị XOR.

In [25]:
def all_subsets(nums):
    res = [[]]
    for v in nums:
        res += [s + [v] for s in res]
    return res

def sum_of_subset_sums(nums):
    return sum(nums) * 2 ** (len(nums) - 1)

print(all_subsets([1, 2, 3]))
print("tổng các tổng tập con [1,2,3] =", sum_of_subset_sums([1, 2, 3]),
      "| liệt kê:", sum(sum(s) for s in all_subsets([1, 2, 3])))

[[], [1], [2], [1, 2], [3], [1, 3], [2, 3], [1, 2, 3]]
tổng các tổng tập con [1,2,3] = 24 | liệt kê: 24


## 15. Bài tập (tự làm trước)

**Bài 1 — tính tay:** liệt kê 4 tập con của `[2, 7]`, XOR từng tập và cộng. Đối chiếu công thức OR.

**Bài 2 — dự đoán output:**

```python
print(6 ^ 3, 6 | 3, 6 & 3, 1 << 4, 13 >> 2 & 1)
```

**Bài 3 — điền TODO:** backtracking chọn/không chọn.

In [26]:
def exercise_subset_xor_sum(nums):
    n = len(nums)
    def dfs(i, cur):
        # TODO 1: nếu i == n trả về cur
        # TODO 2: trả về dfs(bỏ) + dfs(lấy, cur ^ nums[i])
        pass
    return dfs(0, 0)

# assert exercise_subset_xor_sum([1, 3]) == 6
# assert exercise_subset_xor_sum([5, 1, 6]) == 28

**Bài 4 — LeetCode 78:** trả về list mọi tập con (thứ tự bất kỳ) bằng **bitmask**.

In [27]:
def exercise_subsets_bitmask(nums):
    # TODO: for mask in range(1 << n): gom nums[i] khi bit i bật
    pass

# assert sorted(map(sorted, exercise_subsets_bitmask([1, 2, 3]))) == sorted(map(sorted, [[], [1], [2], [3], [1, 2], [1, 3], [2, 3], [1, 2, 3]]))

**Bài 5 — biến thể:** tính **tổng OR** của mọi tập con (tập rỗng có OR = 0).

In [28]:
def exercise_subset_or_sum(nums):
    # TODO: liệt kê tập con (hoặc dùng công thức theo từng bit)
    pass

# assert exercise_subset_or_sum([1, 3]) == 0 + 1 + 3 + 3
# assert exercise_subset_or_sum([5, 1, 6]) == 0 + 5 + 1 + 6 + 5 + 7 + 7 + 7

**Bài 6 — mini challenge:** đếm số tập con (theo chỉ số) có XOR total **bằng `k`**.

In [29]:
def exercise_count_xor_equal(nums, k):
    # TODO: dfs(i, cur) trả về số lá có cur == k; hoặc dict đếm XOR
    pass

# assert exercise_count_xor_equal([1, 2, 3], 0) == 2      # {} và {1,2,3}
# assert exercise_count_xor_equal([2, 2], 2) == 2         # {2a} và {2b}

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | {}, {2}, {7}, {2,7} | `2 ^ 7` theo bit | `0 + 2 + 7 + 5`, OR = 7 |
| 2 | viết nhị phân 6=110, 3=011 | `>>` trước `&` | `13 = 1101` |
| 3 | lá trả `cur` | hai nhánh | `dfs(i+1, cur) + dfs(i+1, cur ^ nums[i])` |
| 4 | mỗi mask một tập | `(mask >> i) & 1` | append list con vào kết quả |
| 5 | thay `^` bằng `|` | liệt kê như bài chính | hoặc theo bit: `2^n - 2^(n-c)` |
| 6 | đếm thay vì cộng | lá trả `1 if cur == k else 0` | cộng hai nhánh |

## 17. Đáp án tham khảo

**Bài 1:** `0 + 2 + 7 + (2^7 = 5) = 14`; công thức: `(2|7) × 2 = 7 × 2 = 14`.

**Bài 2:** `5 7 2 16 1`.

In [30]:
print(subset_xor_sum_bitmask([2, 7]), subset_xor_sum_or([2, 7]))
print(6 ^ 3, 6 | 3, 6 & 3, 1 << 4, 13 >> 2 & 1)

14 14
5 7 2 16 1


In [31]:
def answer_subset_xor_sum(nums):
    n = len(nums)
    def dfs(i, cur):
        if i == n:
            return cur
        return dfs(i + 1, cur) + dfs(i + 1, cur ^ nums[i])
    return dfs(0, 0)


def answer_subsets_bitmask(nums):
    n = len(nums)
    return [[nums[i] for i in range(n) if (mask >> i) & 1] for mask in range(1 << n)]


def answer_subset_or_sum(nums):
    n = len(nums)
    total = 0
    for bit in range(max(nums).bit_length()):
        c = sum(1 for v in nums if (v >> bit) & 1)
        total += (1 << bit) * ((1 << n) - (1 << (n - c)))
    return total


def answer_count_xor_equal(nums, k):
    n = len(nums)
    def dfs(i, cur):
        if i == n:
            return 1 if cur == k else 0
        return dfs(i + 1, cur) + dfs(i + 1, cur ^ nums[i])
    return dfs(0, 0)


assert answer_subset_xor_sum([1, 3]) == 6
assert answer_subset_xor_sum([5, 1, 6]) == 28
assert sorted(map(sorted, answer_subsets_bitmask([1, 2, 3]))) == sorted(map(sorted, [[], [1], [2], [3], [1, 2], [1, 3], [2, 3], [1, 2, 3]]))
assert answer_subset_or_sum([1, 3]) == 7
assert answer_subset_or_sum([5, 1, 6]) == 38
assert answer_count_xor_equal([1, 2, 3], 0) == 2
assert answer_count_xor_equal([2, 2], 2) == 2
for _ in range(200):
    nums = [rng.randint(1, 20) for _ in range(rng.randint(1, 8))]
    brute_or = sum(reduce(lambda a, b: a | b, s, 0) for s in answer_subsets_bitmask(nums))
    assert answer_subset_or_sum(nums) == brute_or
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


`v.bit_length()` = số bit cần để biểu diễn `v` (ví dụ `6 = 110` có `bit_length = 3`). Công thức OR: số tập con có bit `b` = 1 bằng **tất cả** `2^n` trừ đi các tập con chỉ gồm phần tử **không** có bit `b` (`2^(n−c)` tập).

## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def subsetXORSum(self, nums: List[int]) -> int:
        n = len(nums)
        def dfs(i, cur):                     # quyết định nums[i]: bỏ hay lấy
            if i == n:
                return cur                   # một tập con hoàn chỉnh
            return dfs(i + 1, cur) + dfs(i + 1, cur ^ nums[i])
        return dfs(0, 0)
```

Mẹo tắt: `OR(nums) << (n - 1)`.

Khung liệt kê bitmask: `for mask in range(1 << n): for i in range(n): if mask >> i & 1: ...`

Checklist:

- [ ] `^` là XOR, lũy thừa là `**`.
- [ ] Liệt kê theo **chỉ số** (phần tử trùng vẫn đếm riêng).
- [ ] Base case trả `cur`, không phải 0.
- [ ] Trong từng tập con dùng `^`, không dùng `|`.
- [ ] Nếu dùng công thức: dịch trái `n - 1` (không phải `n`).